In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import pandas as pd
import numpy as np
import sqlalchemy as sa
import os
from row_generate_func import *
from auto_insert_func import *

from sqlalchemy.orm import Session

engine = sa.create_engine("postgresql://tongli1997@proddb01.pcic.uvic.ca,proddb02.pcic.uvic.ca/crmp?keepalives=1&keepalives_idle=300&keepalives_interval=300&keepalives_count=9&passfile=/workspaces/crmprtd/.pgpass", echo=False)
session = Session(engine)

In [3]:
# --- Main workflow ---
output_folder = '/workspaces/crmprtd/fern/FERNNorth2025_insert/output/'
file_name = 'Fern_raw_db_station_matched.csv'

df_match = pd.read_csv(output_folder + file_name)

df_match

,station_name,native_id_raw,station_file_name,lat,lon,elev,variable,unit_raw,earliest_time_raw,latest_time_raw,unit_origin,db_var_match,unit_db,earliest_time_db,latest_time_db,earliest_diff_days,latest_diff_days,batch
0,Atlin School,AtlSch,Atlin School,59.574000,-133.687000,705,DewPt,°C,2011-07-28 09:00:00,2024-08-10 19:00:00,°C,DewPtC,celsius,NaN,NaN,NaN,NaN,batch2
1,Atlin School,AtlSch,Atlin School,59.574000,-133.687000,705,Gust Speed,m/s,2010-08-20 19:00:00,2024-08-10 19:00:00,m/s,GustSpeedms,m s-1,NaN,NaN,NaN,NaN,batch2
2,Atlin School,AtlSch,Atlin School,59.574000,-133.687000,705,RH,%,2011-07-28 09:00:00,2024-08-10 19:00:00,%,RH,%,NaN,NaN,NaN,NaN,batch2
3,Atlin School,AtlSch,Atlin School,59.574000,-133.687000,705,Rain,mm,2010-08-20 20:00:00,2024-08-10 19:00:00,mm,Rainmm,mm,NaN,NaN,NaN,NaN,batch2
4,Atlin School,AtlSch,Atlin School,59.574000,-133.687000,705,Sfc Temp,°C,2010-08-20 19:00:00,2024-08-10 19:00:00,°C,Surface_Temp,celsius,NaN,NaN,NaN,NaN,batch4
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
528,Willow-BowronWx,PGTIS2,Willow-BowronWx,53.772577,-122.724634,596,Solar Radiation,W/m²,2007-07-31 15:00:00,2025-10-30 10:00:00,W/m²,SolarRadiationWm,W m-2,2007-07-31 15:00:00,2023-10-17 09:00:00,0.0,744.0,batch1
529,Willow-BowronWx,PGTIS2,Willow-BowronWx,53.772577,-122.724634,596,Temp,°C,2007-07-31 15:00:00,2025-10-30 10:00:00,°C,TempC,celsius,2007-07-31 15:00:00,2023-10-17 09:00:00,0.0,744.0,batch1
530,Willow-BowronWx,PGTIS2,Willow-BowronWx,53.772577,-122.724634,596,Wetness,%,2008-06-12 13:00:00,2025-10-30 10:00:00,%,Wetness,%,2008-06-12 13:00:00,2023-10-17 09:00:00,0.0,744.0,batch1
531,Willow-BowronWx,PGTIS2,Willow-BowronWx,53.772577,-122.724634,596,Wind Direction,ø,2007-07-31 15:00:00,2025-10-30 10:00:00,ø,WindDirection,degree,2007-07-31 15:00:00,2023-10-17 09:00:00,0.0,744.0,batch1


In [4]:
station_names = df_match['station_name'].tolist()

query = sa.text("""
    SELECT DISTINCT m.station_name, s.native_id, m.lat, m.lon, m.elev
    FROM meta_history AS m
    JOIN meta_station AS s ON m.station_id = s.station_id
    WHERE s.network_id = 11
      AND m.station_name = ANY(:station_names)
""")

with engine.connect() as conn:
    db_stations = pd.read_sql(query, conn, params={"station_names": station_names})

In [5]:
merged = df_match.merge(
    db_stations,
    on='station_name',
    how='left'
)

match_info = merged.rename(
    columns={
        'native_id': 'native_id_db',
        'lat_y': 'lat_db',
        'lon_y': 'lon_db',
        'elev_y': 'elev_db'
    }
)

match_info = match_info.drop(columns=['native_id_raw', 'lat_x', 'lon_x', 'elev_x'])

match_info

,station_name,station_file_name,variable,unit_raw,earliest_time_raw,latest_time_raw,unit_origin,db_var_match,unit_db,earliest_time_db,latest_time_db,earliest_diff_days,latest_diff_days,batch,native_id_db,lat_db,lon_db,elev_db
0,Atlin School,Atlin School,DewPt,°C,2011-07-28 09:00:00,2024-08-10 19:00:00,°C,DewPtC,celsius,NaN,NaN,NaN,NaN,batch2,AtlSch,59.574000,-133.687000,705.0
1,Atlin School,Atlin School,Gust Speed,m/s,2010-08-20 19:00:00,2024-08-10 19:00:00,m/s,GustSpeedms,m s-1,NaN,NaN,NaN,NaN,batch2,AtlSch,59.574000,-133.687000,705.0
2,Atlin School,Atlin School,RH,%,2011-07-28 09:00:00,2024-08-10 19:00:00,%,RH,%,NaN,NaN,NaN,NaN,batch2,AtlSch,59.574000,-133.687000,705.0
3,Atlin School,Atlin School,Rain,mm,2010-08-20 20:00:00,2024-08-10 19:00:00,mm,Rainmm,mm,NaN,NaN,NaN,NaN,batch2,AtlSch,59.574000,-133.687000,705.0
4,Atlin School,Atlin School,Sfc Temp,°C,2010-08-20 19:00:00,2024-08-10 19:00:00,°C,Surface_Temp,celsius,NaN,NaN,NaN,NaN,batch4,AtlSch,59.574000,-133.687000,705.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
528,Willow-BowronWx,Willow-BowronWx,Solar Radiation,W/m²,2007-07-31 15:00:00,2025-10-30 10:00:00,W/m²,SolarRadiationWm,W m-2,2007-07-31 15:00:00,2023-10-17 09:00:00,0.0,744.0,batch1,PGTIS2,53.772577,-122.724634,596.0
529,Willow-BowronWx,Willow-BowronWx,Temp,°C,2007-07-31 15:00:00,2025-10-30 10:00:00,°C,TempC,celsius,2007-07-31 15:00:00,2023-10-17 09:00:00,0.0,744.0,batch1,PGTIS2,53.772577,-122.724634,596.0
530,Willow-BowronWx,Willow-BowronWx,Wetness,%,2008-06-12 13:00:00,2025-10-30 10:00:00,%,Wetness,%,2008-06-12 13:00:00,2023-10-17 09:00:00,0.0,744.0,batch1,PGTIS2,53.772577,-122.724634,596.0
531,Willow-BowronWx,Willow-BowronWx,Wind Direction,ø,2007-07-31 15:00:00,2025-10-30 10:00:00,ø,WindDirection,degree,2007-07-31 15:00:00,2023-10-17 09:00:00,0.0,744.0,batch1,PGTIS2,53.772577,-122.724634,596.0


### Batch 1, for the unique variables (only one sensor)

In [7]:
match_info_uniq_var = match_info[match_info['db_var_match'].notna()]
match_info_uniq_var = match_info_uniq_var.reset_index(drop=True)
match_info_uniq_var

,station_name,station_file_name,variable,unit_raw,earliest_time_raw,latest_time_raw,unit_origin,db_var_match,unit_db,earliest_time_db,latest_time_db,earliest_diff_days,latest_diff_days,batch,native_id_db,lat_db,lon_db,elev_db
0,Atlin School,Atlin School,DewPt,°C,2011-07-28 09:00:00,2024-08-10 19:00:00,°C,DewPtC,celsius,NaN,NaN,NaN,NaN,batch2,AtlSch,59.574000,-133.687000,705.0
1,Atlin School,Atlin School,Gust Speed,m/s,2010-08-20 19:00:00,2024-08-10 19:00:00,m/s,GustSpeedms,m s-1,NaN,NaN,NaN,NaN,batch2,AtlSch,59.574000,-133.687000,705.0
2,Atlin School,Atlin School,RH,%,2011-07-28 09:00:00,2024-08-10 19:00:00,%,RH,%,NaN,NaN,NaN,NaN,batch2,AtlSch,59.574000,-133.687000,705.0
3,Atlin School,Atlin School,Rain,mm,2010-08-20 20:00:00,2024-08-10 19:00:00,mm,Rainmm,mm,NaN,NaN,NaN,NaN,batch2,AtlSch,59.574000,-133.687000,705.0
4,Atlin School,Atlin School,Sfc Temp,°C,2010-08-20 19:00:00,2024-08-10 19:00:00,°C,Surface_Temp,celsius,NaN,NaN,NaN,NaN,batch4,AtlSch,59.574000,-133.687000,705.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
471,Willow-BowronWx,Willow-BowronWx,Solar Radiation,W/m²,2007-07-31 15:00:00,2025-10-30 10:00:00,W/m²,SolarRadiationWm,W m-2,2007-07-31 15:00:00,2023-10-17 09:00:00,0.0,744.0,batch1,PGTIS2,53.772577,-122.724634,596.0
472,Willow-BowronWx,Willow-BowronWx,Temp,°C,2007-07-31 15:00:00,2025-10-30 10:00:00,°C,TempC,celsius,2007-07-31 15:00:00,2023-10-17 09:00:00,0.0,744.0,batch1,PGTIS2,53.772577,-122.724634,596.0
473,Willow-BowronWx,Willow-BowronWx,Wetness,%,2008-06-12 13:00:00,2025-10-30 10:00:00,%,Wetness,%,2008-06-12 13:00:00,2023-10-17 09:00:00,0.0,744.0,batch1,PGTIS2,53.772577,-122.724634,596.0
474,Willow-BowronWx,Willow-BowronWx,Wind Direction,ø,2007-07-31 15:00:00,2025-10-30 10:00:00,ø,WindDirection,degree,2007-07-31 15:00:00,2023-10-17 09:00:00,0.0,744.0,batch1,PGTIS2,53.772577,-122.724634,596.0


### Generate and insert data row by row (which means for specific station and variable)

In [10]:
data_path = '/workspaces/crmprtd/fern/FERNNorth2025_VF_data'
db_url = "postgresql://tongli1997@proddb01.pcic.uvic.ca,proddb02.pcic.uvic.ca/crmp?keepalives=1&keepalives_idle=300&keepalives_interval=300&keepalives_count=9&passfile=/workspaces/crmprtd/.pgpass"

for i, row in match_info_uniq_var.iterrows():
    station = row["station_name"]
    variable = row["variable"]

    print(f"\n🔍 [{i+1}/{len(match_info_uniq_var)}] Processing {station} — {variable}")

    # Generate new rows for this station/variable
    new_rows = extract_new_data_rows(row, data_path)
    new_rows = [r for r in new_rows if pd.notna(r.time)]

    if len(new_rows) > 0:
        print(f"  ✅ {len(new_rows)} new records found outside DB time range.")
        
        # Insert immediately to avoid huge memory usage
        safe_insert_rows(
            new_rows,
            log_file_path=output_folder + 'fern_all_station_insert.log',
            db_url=db_url,
            chunk_size=5000,
            bulk_chunk_size=1000,
        )
    else:
        print("  ⚠️ No new records found (matches DB time range exactly).")


🔍 [1/476] Processing Atlin School — DewPt
  ✅ 111039 new records found outside DB time range.
🚀 Starting insert: 111039 rows in 23 chunks
➡️  Processing rows 0–4999
{"asctime": "2026-02-27 01:14:21,520", "levelname": "INFO", "name": "crmprtd.insert", "message": "Using Bulk Insert Strategy"}
{"asctime": "2026-02-27 01:14:22,932", "levelname": "INFO", "name": "crmprtd.insert", "message": "Bulk insert progress: 1000 inserted, 0 skipped, 0 failed"}
{"asctime": "2026-02-27 01:14:24,258", "levelname": "INFO", "name": "crmprtd.insert", "message": "Bulk insert progress: 2000 inserted, 0 skipped, 0 failed"}
{"asctime": "2026-02-27 01:14:25,842", "levelname": "INFO", "name": "crmprtd.insert", "message": "Bulk insert progress: 3000 inserted, 0 skipped, 0 failed"}
{"asctime": "2026-02-27 01:14:27,283", "levelname": "INFO", "name": "crmprtd.insert", "message": "Bulk insert progress: 4000 inserted, 0 skipped, 0 failed"}
{"asctime": "2026-02-27 01:14:28,650", "levelname": "INFO", "name": "crmprtd.in

### Code done

### old code, because 2 years here, too large

In [ ]:
# data_path = '/workspaces/crmprtd/fern/FERNNorth2025_VF_data'

# # subset = match_info_uniq_var.loc[
# #     (match_info_uniq_var['variable'] == 'Soil temp')
# # ]

# all_rows = []

# for i, row in match_info_uniq_var.iterrows():
#     station = row["station_name"]
#     variable = row["variable"]

#     print(f"\n🔍 [{i+1}/{len(match_info_uniq_var)}] Processing {station} — {variable}")

#     new_rows = extract_new_data_rows(row, data_path)
#     all_rows.extend(new_rows)

#     if len(new_rows) > 0:
#         print(f"  ✅ {len(new_rows)} new records found outside DB time range.")
#     else:
#         print("  ⚠️ No new records found (matches DB time range exactly).")

# all_rows = [r for r in all_rows if not pd.isna(r.time)]


🔍 [1/476] Processing Atlin School — DewPt
  ✅ 111039 new records found outside DB time range.

🔍 [2/476] Processing Atlin School — Gust Speed
  ✅ 119237 new records found outside DB time range.

🔍 [3/476] Processing Atlin School — RH
  ✅ 111039 new records found outside DB time range.

🔍 [4/476] Processing Atlin School — Rain
  ✅ 119231 new records found outside DB time range.

🔍 [5/476] Processing Atlin School — Sfc Temp
  ✅ 119237 new records found outside DB time range.

🔍 [6/476] Processing Atlin School — Soil Temp
  ✅ 119237 new records found outside DB time range.

🔍 [7/476] Processing Atlin School — Soil Temp 50 cm
  ✅ 119237 new records found outside DB time range.

🔍 [8/476] Processing Atlin School — Soil Temp 75 cm
  ✅ 119237 new records found outside DB time range.

🔍 [9/476] Processing Atlin School — Solar Radiation
  ✅ 119237 new records found outside DB time range.

🔍 [10/476] Processing Atlin School — Temp
  ✅ 119237 new records found outside DB time range.

🔍 [12/476] 

: 

: 

### Insert

In [ ]:
# len(all_rows)

13121902

In [ ]:
# db_url = "postgresql://tongli1997@proddb01.pcic.uvic.ca,proddb02.pcic.uvic.ca/crmp?keepalives=1&keepalives_idle=300&keepalives_interval=300&keepalives_count=9&passfile=/workspaces/crmprtd/.pgpass"

# safe_insert_rows(
#     all_rows,
#     log_file_path=output_folder + 'fern_all_station_insert.log',
#     db_url=db_url,
#     chunk_size=5000,        # SAFE starting point
#     bulk_chunk_size=1000,   # internal DB bulk insert size
# )

🚀 Starting insert: 13121902 rows in 2625 chunks
➡️  Processing rows 0–4999
{"asctime": "2026-02-24 22:49:29,613", "levelname": "INFO", "name": "crmprtd.insert", "message": "Using Bulk Insert Strategy"}
{"asctime": "2026-02-24 22:49:30,134", "levelname": "INFO", "name": "crmprtd.insert", "message": "Bulk insert progress: 1000 inserted, 0 skipped, 0 failed"}
{"asctime": "2026-02-24 22:49:30,619", "levelname": "INFO", "name": "crmprtd.insert", "message": "Bulk insert progress: 2000 inserted, 0 skipped, 0 failed"}
{"asctime": "2026-02-24 22:49:31,118", "levelname": "INFO", "name": "crmprtd.insert", "message": "Bulk insert progress: 3000 inserted, 0 skipped, 0 failed"}
{"asctime": "2026-02-24 22:49:31,583", "levelname": "INFO", "name": "crmprtd.insert", "message": "Bulk insert progress: 4000 inserted, 0 skipped, 0 failed"}
{"asctime": "2026-02-24 22:49:32,030", "levelname": "INFO", "name": "crmprtd.insert", "message": "Bulk insert progress: 5000 inserted, 0 skipped, 0 failed"}
{"asctime": "